# Extreme OOD Probe Suite — LLaVA-OneVision-1.5

Run **LLaVA-OneVision-1.5-8B** on 7 extreme out-of-distribution probes.

**Environment:** `conda activate llava-ov`

**Before running inference:** generate synthetic images with `python generate_images.py` (see below).

**Output:** `llava_ov_extreme_results.json`


## Prepare images

```bash
cd mywork/extreme_case
python generate_images.py
```

Writes `pure_noise.jpg`, `solid_color.jpg`, `geometric_circle.jpg`, and `test_noisy.jpg` to `mywork/test_images/`.

In [1]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import torch

# ==================== Configuration ====================
# NOTEBOOK_DIR: run from mywork/extreme_case/
# IMAGE_DIR:    synthetic probe images (generate_images.py → mywork/test_images/)
# MODEL_PATH:   LLaVA-OneVision-1.5-8B-Instruct checkpoint
# OUTPUT_JSON:  consumed by compare_extreme_results.py
# transformers==4.53.0 required for correct inference (see model loading helpers)

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
IMAGE_DIR = NOTEBOOK_DIR.parent / "test_images"
MODEL_PATH = (PROJECT_ROOT / "LLaVA-RLHF/models/LLaVA-OneVision-1.5-8B-Instruct").resolve()
OUTPUT_JSON = NOTEBOOK_DIR / "llava_ov_extreme_results.json"

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Image dir:    {IMAGE_DIR}")
print(f"Model path:   {MODEL_PATH}")


Notebook dir: /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/extreme_case
Project root: /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF
Image dir:    /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/test_images
Model path:   /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/LLaVA-RLHF/models/LLaVA-OneVision-1.5-8B-Instruct


In [2]:
# ==================== Model loading helpers ====================
# OneVision uses HuggingFace AutoModel + Qwen-VL chat template. transformers>=5.0 can
# load the checkpoint after patching missing APIs, but produces garbage logits — we
# hard-require 4.53.0 (the version the checkpoint was built for).


def apply_transformers_compat_patches() -> str:
    """Patch missing transformers>=5 APIs so the model can at least be imported.

    These patches are NOT sufficient for correct inference; they only unblock loading.

    Returns:
        Installed transformers version string.
    """
    import transformers

    version = transformers.__version__
    major = int(version.split(".", maxsplit=1)[0])

    if major >= 5:
        import torch.nn as nn
        import transformers.cache_utils as cache_utils
        import transformers.configuration_utils as configuration_utils
        import transformers.modeling_rope_utils as rope_utils
        from transformers.modeling_utils import PreTrainedModel

        if not hasattr(cache_utils, "SlidingWindowCache"):
            cache_utils.SlidingWindowCache = cache_utils.StaticCache

        if not hasattr(configuration_utils, "layer_type_validation"):

            def _layer_type_validation(layer_types):
                """No-op stub for transformers>=5 configuration_utils.layer_type_validation."""
                return layer_types

            configuration_utils.layer_type_validation = _layer_type_validation

        if not hasattr(PreTrainedModel, "set_submodule"):

            def _set_submodule(self, target: str, module: nn.Module) -> None:
                """Assign a nested submodule by dotted path (PreTrainedModel compat)."""
                atoms = target.split(".")
                if len(atoms) == 1:
                    setattr(self, atoms[0], module)
                    return
                parent = self.get_submodule(".".join(atoms[:-1]))
                setattr(parent, atoms[-1], module)

            PreTrainedModel.set_submodule = _set_submodule

        if "default" not in rope_utils.ROPE_INIT_FUNCTIONS:

            def _compute_default_rope_parameters(config, device=None, seq_len=None, **kwargs):
                """Default RoPE frequencies for checkpoints expecting ROPE_INIT_FUNCTIONS['default']."""
                base = config.rope_theta
                dim = getattr(config, "head_dim", None) or (
                    config.hidden_size // config.num_attention_heads
                )
                inv_freq = 1.0 / (
                    base
                    ** (
                        torch.arange(0, dim, 2, dtype=torch.int64)
                        .to(device=device, dtype=torch.float)
                        / dim
                    )
                )
                return inv_freq, 1.0

            rope_utils.ROPE_INIT_FUNCTIONS["default"] = _compute_default_rope_parameters

        warnings.warn(
            "transformers>=5.0 can load this checkpoint but produces invalid outputs. "
            "For working inference run: pip install 'transformers==4.53.0'",
            stacklevel=2,
        )

    return version


def load_config(model_path: str):
    """Load OneVision AutoConfig and ensure text_config.pad_token_id is set.

    Args:
        model_path: Local path to the checkpoint directory.

    Returns:
        Loaded AutoConfig with pad_token_id populated when missing.
    """
    from transformers import AutoConfig

    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    if not hasattr(config.text_config, "pad_token_id") or config.text_config.pad_token_id is None:
        with open(os.path.join(model_path, "generation_config.json")) as f:
            config.text_config.pad_token_id = json.load(f)["pad_token_id"]
    return config


def count_meta_parameters(model) -> int:
    """Count parameters still on the meta device (partial load / OOM signal).

    Args:
        model: Loaded HuggingFace model.

    Returns:
        Number of parameters with device.type == \"meta\".
    """
    return sum(1 for _, param in model.named_parameters() if param.device.type == "meta")


def build_load_kwargs(model_path: str, config, use_quantization: bool, transformers_major: int):
    """Build kwargs for AutoModelForCausalLM.from_pretrained.

    Args:
        model_path: Checkpoint directory path.
        config: Config from load_config.
        use_quantization: If True, attach 4-bit BitsAndBytesConfig.
        transformers_major: Major transformers version (dtype kw name differs).

    Returns:
        Dict of keyword arguments for from_pretrained.
    """
    kwargs = {
        "pretrained_model_name_or_path": model_path,
        "config": config,
        "device_map": "auto",
        "trust_remote_code": True,
    }
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if use_quantization:
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    if transformers_major >= 5:
        kwargs["dtype"] = compute_dtype
    else:
        kwargs["torch_dtype"] = compute_dtype
    return kwargs


def load_ov_model_and_processor(model_path: str):
    """Load LLaVA-OneVision model and processor with version guard and 4-bit fallback.

    Args:
        model_path: Path to the OneVision instruct checkpoint.

    Returns:
        Tuple (model, processor) with model in eval mode.
    """
    from transformers import AutoModelForCausalLM, AutoProcessor

    transformers_version = apply_transformers_compat_patches()
    transformers_major = int(transformers_version.split(".", maxsplit=1)[0])
    if transformers_major >= 5:
        raise RuntimeError(
            "transformers>=5.0 is not supported for inference. "
            "Run: pip install 'transformers==4.53.0'"
        )

    config = load_config(model_path)
    dtype_name = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
    print(f"Loading LLaVA-OneVision model ({dtype_name})...")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            model = AutoModelForCausalLM.from_pretrained(
                **build_load_kwargs(model_path, config, use_quantization=False, transformers_major=transformers_major)
            )
            if count_meta_parameters(model):
                raise torch.cuda.OutOfMemoryError("weights left on meta device")
        except torch.cuda.OutOfMemoryError:
            print("[WARN] Insufficient VRAM; retrying with 4-bit quantization...")
            torch.cuda.empty_cache()
            model = AutoModelForCausalLM.from_pretrained(
                **build_load_kwargs(model_path, config, use_quantization=True, transformers_major=transformers_major)
            )

    model.eval()
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    print("[OK] Model and processor ready")
    return model, processor


In [3]:
# ==================== Probe definitions & scoring (schema v1.0) ====================
# Categories: A=noise, B=solid color, C=perturbed real image, D=geometry
# A2 is qualitative (manual review only); scored accuracy excludes A2 (6 probes).
QUALITATIVE_IDS = {"A2"}

PROBES = [
    {
        "image": "pure_noise.jpg",
        "id": "A1",
        "type": "existence",
        "question": "Is this a photograph of a real scene? Answer yes or no.",
        "gt": "no",
        "note": "Pure noise has no real scene; checks scene fabrication on zero semantics",
    },
    {
        "image": "pure_noise.jpg",
        "id": "A2",
        "type": "qualitative",
        "question": "Describe this image in detail.",
        "gt": "noise",
        "note": "Qualitative only: inspect whether noise is narrativized into objects (not auto-scored)",
    },
    {
        "image": "solid_color.jpg",
        "id": "B1",
        "type": "attribute",
        "question": "What color is this? Answer with one word.",
        "gt": "blue",
        "note": "Flat steelblue field; basic color sanity check",
    },
    {
        "image": "solid_color.jpg",
        "id": "B2",
        "type": "existence",
        "question": "Is there a sky in this image? Answer yes or no.",
        "gt": "no",
        "note": "Solid color must not be projected into a sky scene",
    },
    {
        "image": "test_noisy.jpg",
        "id": "C1",
        "type": "count",
        "question": "How many tissue boxes are in this image? Answer with a number.",
        "gt": "1",
        "note": "Light noise on test.jpg; count should remain 1",
    },
    {
        "image": "test_noisy.jpg",
        "id": "C2",
        "type": "existence",
        "question": "Is there a keyboard in this image? Answer yes or no.",
        "gt": "no",
        "note": "Noisy office scene; keyboard context trap should stay negative",
    },
    {
        "image": "geometric_circle.jpg",
        "id": "D1",
        "type": "existence",
        "question": "Is this a drawing of an animal? Answer yes or no.",
        "gt": "no",
        "note": "Simple circle; checks sun/animal shape priors",
    },
]

SCHEMA_VERSION = "1.0"
BENCHMARK_NAME = "extreme_ood_probe"


def judge_correctness(pred, gt, qtype):
    """Score a model answer against ground truth for auto-graded probes.

    Args:
        pred: Model-generated answer text.
        gt: Ground-truth string (e.g. yes/no, color, count).
        qtype: Probe type (qualitative probes return None).

    Returns:
        True if correct, False if wrong, or None for qualitative probes.
    """
    pred_lower = pred.lower().strip(".")
    gt_lower = gt.lower().strip()
    if qtype == "qualitative":
        return None
    if gt_lower in ["yes", "no"]:
        if gt_lower == "yes":
            return "yes" in pred_lower and "no" not in pred_lower
        return "no" in pred_lower and "yes" not in pred_lower
    return gt_lower in pred_lower


def _verdict_label(correct, qtype, probe_id):
    """Map correctness to a human-readable verdict label.

    Args:
        correct: Result from judge_correctness (or None).
        qtype: Probe type string.
        probe_id: Probe id (e.g. A2 is always qualitative).

    Returns:
        One of "qualitative", "n/a", "correct", or "wrong".
    """
    if probe_id in QUALITATIVE_IDS or qtype == "qualitative":
        return "qualitative"
    if correct is None:
        return "n/a"
    return "correct" if correct else "wrong"


def build_benchmark_output(run_id, models, probes, answers_by_model):
    """Assemble the v1.0 benchmark JSON structure from raw answers.

    Args:
        run_id: Identifier for this run (e.g. llava_rlhf_extreme).
        models: List of model metadata dicts with "key", "name", "path".
        probes: Probe definition list (PROBES).
        answers_by_model: Dict mapping model key -> probe id -> answer text.

    Returns:
        Dict ready for JSON serialization (schema_version, summary, per_probe, ...).
    """
    model_keys = [m["key"] for m in models]
    per_probe = []
    for probe in probes:
        answers, correct, verdict = {}, {}, {}
        for key in model_keys:
            ans = answers_by_model[key].get(probe["id"], "")
            ok = judge_correctness(ans, probe["gt"], probe["type"])
            answers[key] = ans
            correct[key] = ok
            verdict[key] = _verdict_label(ok, probe["type"], probe["id"])
        per_probe.append({**probe, "answers": answers, "correct": correct, "verdict": verdict})

    by_model = {}
    for key in model_keys:
        scored = [r for r in per_probe if r["id"] not in QUALITATIVE_IDS]
        c = sum(1 for r in scored if r["correct"][key])
        t = len(scored)
        by_type = {}
        for r in per_probe:
            if r["id"] in QUALITATIVE_IDS:
                continue
            q = r["type"]
            by_type.setdefault(q, {"correct": 0, "wrong": 0})
            if r["correct"][key]:
                by_type[q]["correct"] += 1
            else:
                by_type[q]["wrong"] += 1
        by_model[key] = {
            "correct": c,
            "wrong": t - c,
            "accuracy": round(c / max(t, 1), 4),
            "qualitative_ids": sorted(QUALITATIVE_IDS),
            "by_type": by_type,
        }

    summary = {
        "total": len(per_probe),
        "scored_total": len([p for p in probes if p["id"] not in QUALITATIVE_IDS]),
        "by_model": by_model,
    }

    return {
        "schema_version": SCHEMA_VERSION,
        "benchmark": BENCHMARK_NAME,
        "run_id": run_id,
        "models": models,
        "probes": probes,
        "per_probe": per_probe,
        "summary": summary,
    }


def save_benchmark_output(output, path):
    """Write benchmark dict to a UTF-8 JSON file.

    Args:
        output: Dict from build_benchmark_output.
        path: Filesystem path for the output .json file.

    Returns:
        None.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)


In [4]:
# ==================== Inference ====================

from qwen_vl_utils import process_vision_info


def generate_answer_ov(model, processor, image_path: str, question: str) -> str:
    """Run greedy OneVision generation for one (image, question) pair.

    Args:
        model: Loaded AutoModelForCausalLM (OneVision).
        processor: Matching AutoProcessor with chat template.
        image_path: Path to probe image file.
        question: User question text.

    Returns:
        Decoded assistant reply string (prompt tokens trimmed).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]

    # Qwen-VL chat template; add_generation_prompt=True opens the assistant turn
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    max_new_tokens = 128 if question.startswith("Describe") else 32
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only tokens generated after the prompt (per batch row)
    trimmed = [
        out_ids[len(in_ids) :]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


In [5]:
# ==================== Load model, run probes, save JSON ====================

model, processor = load_ov_model_and_processor(str(MODEL_PATH))

print("=" * 80)
print("Running LLaVA-OneVision on extreme OOD probes...")
print("=" * 80)

results_ov = {}
for p in PROBES:
    img_path = IMAGE_DIR / p["image"]
    if not img_path.is_file():
        print(f"[SKIP] {img_path} not found — run generate_images.py first")
        continue

    ans = generate_answer_ov(model, processor, str(img_path), p["question"])
    results_ov[p["id"]] = ans

    if p["id"] in QUALITATIVE_IDS:
        print(f"[OV][{p['id']}] {p['image'][:18]:<18} | (qualitative) | {ans[:80]}")
        continue

    ok = judge_correctness(ans, p["gt"], p["type"])
    status = "OK" if ok else "WRONG"
    print(f"[OV][{p['id']}] {p['image'][:18]:<18} | {status:<5} | {ans[:60]}")

benchmark_output = build_benchmark_output(
    run_id="llava_ov_extreme",
    models=[
        {
            "key": "llava_ov",
            "name": "LLaVA-OneVision-1.5-8B-Instruct",
            "path": str(MODEL_PATH),
        }
    ],
    probes=PROBES,
    answers_by_model={"llava_ov": results_ov},
)
per_probe = benchmark_output["per_probe"]
stats = benchmark_output["summary"]["by_model"]["llava_ov"]

print("\n" + "=" * 100)
print("LLaVA-OneVision EXTREME OOD PROBE RESULTS")
print("=" * 100)
print(f"{'ID':<4} {'Image':<18} {'Type':<12} {'GT':<10} {'Answer':<35} {'Verdict'}")
print("-" * 100)
for row in per_probe:
    ans = row["answers"]["llava_ov"]
    verdict = row["verdict"]["llava_ov"]
    print(
        f"{row['id']:<4} {row['image'][:17]:<18} {row['type']:<12} "
        f"{row['gt']:<10} {ans[:34]:<35} {verdict}"
    )
print("-" * 100)
print(
    f"\nScored accuracy (A2 excluded): {stats['correct']}/{stats['correct'] + stats['wrong']} "
    f"({stats['accuracy']:.1%})"
)
print("=" * 100)

save_benchmark_output(benchmark_output, OUTPUT_JSON)
print(f"\n[OK] Saved {OUTPUT_JSON}")


/data/workspaces/zwang829/conda_envs/llava-ov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading LLaVA-OneVision model (bf16)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.59it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OK] Model and processor ready
Running LLaVA-OneVision on extreme OOD probes...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

[OV][A2] pure_noise.jpg     | (qualitative) | The image is a digital representation of what appears to be static or noise, com
[OV][B1] solid_color.jpg    | OK    | blue
[OV][B2] solid_color.jpg    | WRONG | Yes


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][C1] test_noisy.jpg     | OK    | 1


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][C2] test_noisy.jpg     | OK    | No
[OV][D1] geometric_circle.j | OK    | No, this is not a drawing of an animal. It is a simple black

LLaVA-OneVision EXTREME OOD PROBE RESULTS
ID   Image              Type         GT         Answer                              Verdict
----------------------------------------------------------------------------------------------------
A1   pure_noise.jpg     existence    no         No, this is not a photograph of a   correct
A2   pure_noise.jpg     qualitative  noise      The image is a digital representat  qualitative
B1   solid_color.jpg    attribute    blue       blue                                correct
B2   solid_color.jpg    existence    no         Yes                                 wrong
C1   test_noisy.jpg     count        1          1                                   correct
C2   test_noisy.jpg     existence    no         No                                  correct
D1   geometric_circle.  existence    no         No, this is not a draw

## Next steps

After **`llava_rlhf_extreme_results.json`** (from `extreme_rlhf.ipynb`) and **`llava_ov_extreme_results.json`** (this notebook) both exist, run:

```bash
python compare_extreme_results.py
```
